In [2]:
import os
os.listdir("/kaggle/input")

['datasets']

In [3]:
print(os.listdir("/kaggle/input/datasets/varun000reddy/training/train/i"))

['annos', 'image']


In [18]:
import os
import json
from collections import Counter
from tqdm import tqdm

# Path to annotations folder
annos_path = "/kaggle/input/datasets/varun000reddy/training/train/annos"

category_counter = Counter()

# Loop through all annotation files
for file in tqdm(os.listdir(annos_path)):
    if file.endswith(".json"):
        with open(os.path.join(annos_path, file)) as f:
            data = json.load(f)
            
            # Each clothing item is stored as item1, item2, ...
            for key in data:
                if key.startswith("item"):
                    category_id = data[key]["category_id"]
                    category_counter[category_id] += 1

# Show all categories sorted
print("Category frequencies:")
for cat, count in category_counter.most_common():
    print(f"Category {cat} → {count}")

100%|██████████| 191961/191961 [08:21<00:00, 382.62it/s]

Category frequencies:
Category 1 → 71645
Category 8 → 55387
Category 7 → 36616
Category 2 → 36064
Category 9 → 30835
Category 12 → 17949
Category 10 → 17211
Category 5 → 16095
Category 4 → 13457
Category 11 → 7907
Category 13 → 6492
Category 6 → 1985
Category 3 → 543


In [19]:
top5 = category_counter.most_common(5)

print("\nTop 5 Categories:")
for cat, count in top5:
    print(f"Category {cat} → {count}")


Top 5 Categories:
Category 1 → 71645
Category 8 → 55387
Category 7 → 36616
Category 2 → 36064
Category 9 → 30835


In [20]:
category_map = {
    1: "short sleeve top",
    2: "long sleeve top",
    3: "short sleeve outwear",
    4: "long sleeve outwear",
    5: "vest",
    6: "sling",
    7: "shorts",
    8: "trousers",
    9: "skirt",
    10: "short sleeve dress",
    11: "long sleeve dress",
    12: "vest dress",
    13: "sling dress"
}

print("\nTop 5 Category Names:")
for cat_id, count in top5:
    print(f"{category_map[cat_id]} → {count}")


Top 5 Category Names:
short sleeve top → 71645
trousers → 55387
shorts → 36616
long sleeve top → 36064
skirt → 30835


In [4]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2026-03-03 04:28:28.091126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772512108.325260      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772512108.399127      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772512108.962972      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772512108.963009      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772512108.963013      55 computation_placer.cc:177] computation placer alr

In [5]:
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detected:", gpus)

GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [6]:
top5_ids = [1, 8, 7, 2, 9]
num_classes = 5

In [7]:
def load_filtered_dataset(image_dir, anno_dir, top5_ids):

    images = []
    labels = []

    json_files = sorted(os.listdir(anno_dir))

    for file in json_files:
        if not file.endswith(".json"):
            continue

        img_name = file.replace(".json", ".jpg")
        img_path = os.path.join(image_dir, img_name)

        

        label = np.zeros(len(top5_ids))
        contains_top5 = False

        with open(os.path.join(anno_dir, file)) as f:
            data = json.load(f)

        for key in data:
            if key.startswith("item"):
                cat_id = data[key]["category_id"]
                if cat_id in top5_ids:
                    idx = top5_ids.index(cat_id)
                    label[idx] = 1
                    contains_top5 = True

        if contains_top5:
            images.append(img_path)
            labels.append(label)

    print("Filtered dataset size:", len(images))
    return np.array(images), np.array(labels)

In [8]:
train_img_dir = "/kaggle/input/datasets/varun000reddy/training/train/image"
train_anno_dir = "/kaggle/input/datasets/varun000reddy/training/train/annos"

val_img_dir = "/kaggle/input/datasets/varun000reddy/validation/validation/image"
val_anno_dir = "/kaggle/input/datasets/varun000reddy/validation/validation/annos"

train_images, train_labels = load_filtered_dataset(
    train_img_dir, train_anno_dir, top5_ids
)

val_images, val_labels = load_filtered_dataset(
    val_img_dir, val_anno_dir, top5_ids
)

Filtered dataset size: 144174
Filtered dataset size: 23741


In [9]:
class_counts = np.sum(train_labels, axis=0)
total_samples = len(train_labels)

pos_weights = (total_samples - class_counts) / (class_counts + 1e-6)

print("Class counts:", class_counts)
print("Positive weights:", pos_weights)

Class counts: [70586. 54969. 36332. 35751. 30625.]
Positive weights: [1.04252968 1.62282377 2.96823737 3.03272636 3.70772245]


In [10]:
@tf.keras.utils.register_keras_serializable()
def weighted_binary_crossentropy(pos_weights):

    pos_weights_tensor = tf.constant(pos_weights, dtype=tf.float32)

    def loss(y_true, y_pred):
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        weights = y_true * pos_weights_tensor + (1 - y_true)
        return tf.reduce_mean(bce * weights)

    return loss

In [11]:
IMG_SIZE = 224
BATCH_SIZE = 128   # Increased for 2 GPUs

def preprocess(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_labels))
val_ds = val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

I0000 00:00:1772514094.303476      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772514094.309720      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [17]:
strategy = tf.distribute.MirroredStrategy()
print("Number of GPUs used:", strategy.num_replicas_in_sync)

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of GPUs used: 2


In [11]:
with strategy.scope():

    base_model = keras.applications.ResNet50(
        include_top=False,
        weights=None,   # FROM SCRATCH
        input_shape=(224, 224, 3)
    )

    x = layers.GlobalAveragePooling2D()(base_model.output)
    output = layers.Dense(num_classes, activation='sigmoid')(x)

    model = keras.Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=weighted_binary_crossentropy(pos_weights),
        metrics=["accuracy"]
    )

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "checkpoint_epoch_{epoch}.keras",
    save_freq="epoch"
)


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[checkpoint]
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1772452251.307884     135 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772452251.316107     132 cuda_dnn.cc:529] Loaded cuDNN version 91002


1127/1127 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2634 - loss: 1.8913INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 1540s 1s/step - accuracy: 0.2635 - loss: 1.8912 - val_accuracy: 0.2472 - val_loss: 2.1933
Epoch 2/40
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 1505s 1s/step - accuracy: 0.3647 - loss: 1.6190 - val_accuracy: 0.3985 - val_loss: 1.7760
Epoch 3/40
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 1515s 1s/step - accuracy: 0.4161 - loss: 1.4360 - val_accuracy: 0.4903 - val_loss: 1.7654
Epoch 4/40
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 1504s 1s/step - accuracy: 0.4560 - loss: 1.2820 - val_accuracy: 0.4477 - val_loss: 1.4537
Epoch 5/40
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 1494s 1s/step - accuracy: 0.4884 - loss: 1.1418 - val_accuracy: 0.3775 - val_loss: 1.3

In [3]:
import os
os.listdir("/kaggle/working")

['.virtual_documents', 'checkpoint_epoch_27.keras']

In [ ]:
keep_file = "checkpoint_epoch_27.keras"

for f in os.listdir("/kaggle/working"):
    if f.startswith("checkpoint_epoch_") and f != keep_file:
        os.remove(os.path.join("/kaggle/working", f))

print("Only epoch 27 kept.")

In [18]:
pos_weights = [1.04252968, 1.62282377, 2.96823737, 3.03272636, 3.70772245]

def weighted_binary_crossentropy(pos_weights):
    pos_weights_tensor = tf.constant(pos_weights, dtype=tf.float32)

    def loss(y_true, y_pred):
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        weights = y_true * pos_weights_tensor + (1 - y_true)
        return tf.reduce_mean(bce * weights)

    return loss

loss_fn = weighted_binary_crossentropy(pos_weights)



In [19]:
with strategy.scope():
    model = tf.keras.models.load_model(
        "checkpoint_epoch_27.keras",
        custom_objects={"loss": loss_fn}
    )

In [20]:
best_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/kaggle/working/best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

In [21]:
history=model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=27,
    epochs=35,
    callbacks=[best_checkpoint]
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1772515405.831654     143 cuda_dnn.cc:529] Loaded cuDNN version 91002


1127/1127 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6776 - loss: 0.1052INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

Epoch 28: val_loss improved from inf to 3.68787, saving model to /kaggle/working/best_model.keras
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 1419s 1s/step - accuracy: 0.6776 - loss: 0.1052 - val_accuracy: 0.4808 - val_loss: 3.6879
Epoch 29/35
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6848 - loss: 0.1057
Epoch 29: val_loss improved from 3.68787 to 3.35443, saving model to /kaggle/working/best_model.keras
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 1383s 1s/step - accuracy: 0.6848 - loss: 0.1057 - val_accuracy: 0.5785 - val_loss: 3.3544
Epoch 30/35
1127/1127 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6853 - loss: 0.0961
Epoch 30: val_loss improve

In [1]:
model.save("/kaggle/working/final_model_35epochs.keras")

NameError: name 'model' is not defined